In [ ]:
#uncomment ONLY on second run 
#can skip the next FOUR cells after running this cell

#df_exploded = spark.read.parquet("gs://bucket-ps3/df_exploded")


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder.appName("NarrativeRedundancy").getOrCreate()

df = spark.read.csv(
    "gs://bucket-ps3/clean_texts.csv",
    header=True,
    multiLine=True, #avoids reading a single line as a row
    escape='"',     #specifies limits of each "line"(cell) as quotation marks
    inferSchema=True
)

df.printSchema()
df.show(3, truncate=100)
print(f"Number of films: {df.count()}")

In [ ]:
from pyspark.sql.functions import udf, col
from pyspark.sql.types import ArrayType, StringType
import re
def split_sentences(text): 
    """function that splits the script text to sentences. 
    Filters out sentences that are one words like 'No.' 
    Limit of sentence length set at >3. Anything with 3
    characters or shorter is dropped. 
    """
    if text is None: #defense from crashing if text is missing
        return []
    sentences = re.split(r'(?<=[.!?])\s+', text)   #split text as sentences on space/s after a punctuation                             
    return [s.strip() for s in sentences if s and  #filter out empty text 
            s.strip() and len(s.strip()) > 3]      #remove whitespace and sentences with less than 3 characters
                                                   #protect confusion of counting abbreviations (ie. "Mr. Joe") and splitting on ("Mr." & " Joe")

#translate python code to pyspark code using UDF
split_sentences_udf = udf(split_sentences, ArrayType(StringType())) #takes argument 'split_sentences' as function to be applied to data later on 
                                                                    #specifies the datatype for the output produced by the function

#create a new column "sentences"; cells = output of the the split_sentences_udf function; new df created and named 'df_split_sentences'
df_split_sentences = df.withColumn("sentences", split_sentences_udf(col("clean_text")))

#sample output
df_split_sentences.select("title", "sentences").show(2, truncate=80, vertical = True)

In [ ]:
#Exploding Column 'Sentences' -> one sentence per row
from pyspark.sql.functions import udf, col, explode

df_exploded = df_split_sentences.select(         #create a new exploded df with the ff columns only
    col('imdb_id'),                              #select imdb_id, title, and sentences columns
    col('title'),                                #keeps each sentence linked to imdb_id (for analysis) and title(for human readability)
    explode(col("sentences")).alias("sentence")) #explode 'sentences' col; rename col to 'sentence'

#sample output
df_exploded.show(5, truncate=80)
print(f"Total sentences: {df_exploded.count()}")

In [ ]:
#CHECKPOINT
#ONLY RUN ON FIRST RUN
#comment out after the first run
df_exploded.write.mode("overwrite").parquet("gs://bucket-ps3/df_exploded")


In [ ]:
#Initial cell executing distributed embedding with pandas_udf on Spark
#Unsuccessful running cell due to GCP issues: No GPU quota available and worker caching permission issues on Dataproc..
#.. keeping each worker from loading SBERT locally

#import pandas as pd
#from sentence_transformers import SentenceTransformer
#from pyspark.sql.functions import pandas_udf
#from pyspark.sql.types import ArrayType, FloatType

#SBERT_MODEL = None  #create global variable for all workers to use and load once. 

#@pandas_udf(ArrayType(FloatType()))                            #specify pandas_udf decorator to enable batch sentence grabbing; specify outut as array of floats(vectors)
#def embeddings(batch: pd.Series) -> pd.Series:                 #specify that input batch is a pandas series; output also a pandas series
#    """creates/assigns vectors for the sentences""" 
#    global SBERT_MODEL                                         #specifies global variable
#    if SBERT_MODEL is None:                                    #loads the SBERT global var ONCE IF not yet loaded
#        SBERT_MODEL = SentenceTransformer('all-MiniLM-L6-v2')  #defines SBERT model used
#    vectors = SBERT_MODEL.encode(batch)                        #defines vectors = sbert model applied to the batches of sentences(batch)
#    return   pd.Series(list(vectors))                          #returns back a pandas series of vectors of the sentences

In [ ]:
# test on 10 sentences first
#test_df = df_exploded.limit(10).withColumn("embedding", embeddings(col("sentence")))
#test_df.select("sentence", "embedding").show(3, truncate=50)

In [ ]:
#Backup Plan - previous processing of all 1.2M sentences crashed GCP ntbk
#Embedding done on master node instead of several worker nodes
#Processing of embeddings done in chunks, saving each chunk everytime, for memory reasons and safer checkpoints
#Debugged version
import pandas as pd
import numpy as np
import time                                            
import subprocess                                       # tool to run commands as a terminal command; gsutil - for saving files to bucket
from sentence_transformers import SentenceTransformer

#Load everything to master node at once
print("Loading data...")
df_exploded = spark.read.parquet("gs://bucket-ps3/df_exploded")    #read saved parquet file from bucket to SparkDF
pandas_df_exploded = df_exploded.toPandas()                        #convert SparkDF to pandasDF
print(f"Total: {len(pandas_df_exploded)} sentences")               #counts how many sentences are in the pandaDF

#Load SBERT model outside for easy reuse on each chunk loop, 
#rather than loading SBERT inside each loop;  avoids unnecessary repetition and memory consumption
print("Loading SBERT...")
SBERT_model = SentenceTransformer('all-MiniLM-L6-v2')

#Specify chunk size and total number of chunks
CHUNK_SIZE = 200000                                  #specify number of sentences per chunk => 200,000 sentences/chunk
total_num_chunks= (len(pandas_df_exploded) + CHUNK_SIZE-1) //CHUNK_SIZE     #calculate how many chunks of 200k needed to cover all sentences (~1.2M)
print(f"Processing {total_num_chunks} chunks of {CHUNK_SIZE}\n")   #total_num_chunks = 6 -> (6x200,000 = 1.2M) 

for i in range(total_num_chunks):   #go through each chunk (6 chunks)
    beginning, ending = i * CHUNK_SIZE, (i+ 1) * CHUNK_SIZE  #work with every 200,000 - ie: 0,200000 -> 200000, 400000 so on..
    chunk = pandas_df_exploded.iloc[beginning:ending]        #specifies what sentences to include in 'chunk' in each loop using beginning,ending set limits on prev line # last loop will be 1,000,000 to 1,200,000 goes past mathematically but allows for exhaustiveness without errors like a normal slice call in python
    print(f"--- Chunk {i+1}/{total_num_chunks} (rows {beginning}–{min(ending, len(pandas_df_exploded))})---")
    #basically print what chunk we are on / (of) total number of chunks. print what rows are being processed (start from the beginning index (ie, 1000000) to the smaller value between the ending index value and the length of the df (1,169,240 < 1,200,000)) output = "rows 1,000,000-1,169,240"

    run_time = time.time()           #shows run time of SBERT for each chunk loop
    chunk_emb = SBERT_model.encode(  #apply SBERT model using .encode()
        chunk['sentence'],           #applied to col 'sentence' inside chunk (a slice of the df)
        batch_size=64,               #commonly used batch size number for SBERT models, good for speed and memory
        show_progress_bar=False,     #don't show progress bar, not needed
        convert_to_numpy=True        #convert output to numpy arrays for efficient saving, reloading, and cosine similarity analysis next
    )
    print(f" Embedded {len(chunk)} in {(time.time()-run_time)/60:.1f} min")
    #print output for human visual readability and easy assessment of processing.. 
    #...Embedded (200000) in (how much time it took to embed 200k (divide by 60 to show as minutes)..
    #.. (:.1f to format value as having only 1 decimal place, and make it a float; for time readability)then 'min' for minutes)

    np.save(f'/tmp/emb_{i:03d}.npy', chunk_emb) #i:03d -> format the variable i with (:) 0s infront of it with a length of 3 characters and d to make it an integer. this is good to make sure file names are in proper order numerically.
                                            #np.save() takes 2 arguments,creates the file name (and tells where to save data) and the data being saved (chunk_emb)
    chunk[['imdb_id', 'title', 'sentence']].to_parquet(f'/tmp/meta_{i:03d}.parquet') #selects the 3 columns and saves or creates a parquet file for the specific chunk


    subprocess.run(['gsutil', 'cp', f'/tmp/emb_{i:03d}.npy',                #copies embedding numpy file from master node to bucket
                f'gs://bucket-ps3/chunks/emb_{i:03d}.npy'], check=True) #check=True to show an error if unsuccessful
    subprocess.run(['gsutil', 'cp', f'/tmp/meta_{i:03d}.parquet',           #copies metadata parquet file from master node to bucket
                f'gs://bucket-ps3/chunks/meta_{i:03d}.parquet'], check=True)
    print(f" ✓ Uploaded chunk {i+1}\n")                                     #visual for readability to confirm all chunk files have been uploaded

    del chunk_emb # delete chunk_emb after each loop so loop starts fresh, memory is cleared and not used up unnecessarily 

In [ ]:
import numpy as np
import pandas as pd
import subprocess
import os

#Creation of folder 'chunks'; upload on master node from bucket
os.makedirs('/tmp/chunks', exist_ok=True) #create a folder called chunks; if folder exists keep going
subprocess.run(['gsutil', '-m', 'cp', 'gs://bucket-ps3/chunks/*', '/tmp/chunks/'], check=True) #uploads/copies temporary 'chunks' file stored from bucket to master node

#Organize embeddings; gather all embedding files into one and combine all embeddings vectors into one complete array (6chunks -> 1 full array)
emb_files = sorted([f for f in os.listdir('/tmp/chunks') if f.
                    startswith('emb_')]) #filters f.startswith() only the files under chunk directory that has embeddings which start with 'emb_'
                                        #sorted() necessary so that during cosine similarity its in order as some movies might have been split across files..
                                        #..also ensures metadata files are perfectly matched up with the right embeddings file to keep data accuracy with..
                                        #..sentences (metadata) paired correctly with the right vectors (embeddings).
embeddings = np.concatenate([np.load(f'/tmp/chunks/{f}') for f in emb_files]) #gather and collect all numpy arrays in each embedding file into one complete numpy array ("embeddings")
print(f"Combined embeddings shape: {embeddings.shape}") #visually confirm the shape of the complete embeddings array

#Do the same organization from embeddings to metadata; using pandas and parquet since its metadata. 
meta_files = sorted([f for f in os.listdir('/tmp/chunks') if f.
                     startswith('meta_')])
metadata = pd.concat([pd.read_parquet(f'/tmp/chunks/{f}') for f in meta_files],
                     ignore_index=True) #make sure index in each file flows through.. 
                                        #..each folder rather than starting at 0 in every..
                                        #..file. keeps indeces matching with indeces in..
                                        #..embeddings file. necessary to keep sentences.. 
                                        #..and vectors matched up perfectly for analysis
print(f"Combined metadata shape: {metadata.shape}")  #visually confirm the shape of the complete metadata file


#Verification of embeddings and metadata matching
assert len(embeddings) == len(metadata), f"Mismatch: {len(embeddings)} vs {len(metadata)}"
#using len() to make sure that the embeddings are accurately matching with the metadata, infers each row in both files have the same appropriate index (realistically: same length)

#save temp files on master node, upload to bucket permanently
np.save('/tmp/embeddings.npy', embeddings) #save embeddings variable to embeddings numpy file temporarily in master node 
metadata.to_parquet('/tmp/metadata.parquet') #save metadata as parquet to temporary metadata file in master node

subprocess.run(['gsutil', 'cp', '/tmp/embeddings.npy', 'gs://bucket-ps3/embeddings.npy'], check=True) #copies the temp embeddings file from master node to bucket permanently
subprocess.run(['gsutil', 'cp', '/tmp/metadata.parquet', 'gs://bucket-ps3/metadata.parquet'], check=True) #copies the temp metadata file from master node to bucket permanently
print("\nDONE - Final combined files in bucket:")           # visual that all tasks are completed
print(" gs://bucket-ps3/embeddings.npy (1169240 x 384)")
print(" gs://bucket-ps3/metadata.parquet (imdb_id, title, sentence)")